# 5 · How a hook is executed — real examples

This demo *fires* each of the six hooks with a concrete payload and a real
callback, and prints the resulting ordered log. It is fully local: we drive the
hook machinery directly, no endpoint required.

The payload type tells you what data each hook can see:

| Event | Payload the callback receives |
|-------|-------------------------------|
| `pretooluse`      | `{ tool, args }` |
| `posttooluse`     | `{ tool, output, is_error }` |
| `userpromptsubmit`| `{ prompt }` |
| `stop`            | `{ reason }` |
| `subagentstart`   | `{ name, prompt }` |
| `subagentstop`    | `{ name, result }` |


In [ ]:
:dep agent_loop = { path = "/home/christian/Sandbox/agent-loop" }
:dep serde_json = "1"

use agent_loop::hooks::{Hooks, HookEvent, HookPayload};
use serde_json::json;

// Enable the shared log so we can replay the exact firing order at the end.
let mut hooks = Hooks::new().with_log();

// Subscribe: one callback per event, each just logs what it saw.
hooks.on_user_prompt_submit(|p| if let HookPayload::UserPromptSubmit { prompt } = p {
    println!("userpromptsubmit: user asked: {:?}", &prompt[..prompt.len().min(40)]);
});
hooks.on_pre_tool_use(|p| if let HookPayload::PreToolUse { tool, args } = p {
    println!("pretooluse: about to call `{tool}` args={args}");
});
hooks.on_post_tool_use(|p| if let HookPayload::PostToolUse { tool, output, is_error } = p {
    let frag: String = output.chars().take(40).collect();
    println!("posttooluse: `{tool}` is_error={is_error} -> {frag}…");
});
hooks.on_stop(|p| if let HookPayload::Stop { reason } = p {
    println!("stop: turn finished with {reason}");
});
hooks.on_sub_agent_start(|p| if let HookPayload::SubAgentStart { name, .. } = p {
    println!("subagentstart: spawning sub-agent `{name}`");
});
hooks.on_sub_agent_stop(|p| if let HookPayload::SubAgentStop { name, result } = p {
    let frag: String = result.chars().take(30).collect();
    println!("subagentstop: sub-agent `{name}` returned: {frag}…");
});
println!("attached callbacks: {}", hooks.count());


### 5.1 · `userpromptsubmit`

Fired the instant a user message enters the loop — the natural place for
approval prompts, input scrubbing, or logging "who asked what".


In [ ]:

hooks.fire(HookEvent::UserPromptSubmit, &HookPayload::UserPromptSubmit {
    prompt: "Summarize this repo and fix the build".into(),
});


### 5.2 · `pretooluse`

Fired *before* a tool runs. It carries the tool name and the parsed arguments —
the right place for a guardrail that can veto an expensive or dangerous call.


In [ ]:

hooks.fire(HookEvent::PreToolUse, &HookPayload::PreToolUse {
    tool: "bash_run".into(),
    args: json!({"command": "rm -rf /tmp/scratch", "timeout_secs": 30}),
});


### 5.3 · `posttooluse`

Fired *after* a tool returns. It carries the output (and whether it errored) —
the natural place for telemetry, error alerting, or caching results.


In [ ]:

hooks.fire(HookEvent::PostToolUse, &HookPayload::PostToolUse {
    tool: "bash_run".into(),
    output: "build finished in 4.2s, tests passed".into(),
    is_error: false,
});


### 5.4 · `stop`

Fired when the turn ends (`finish_reason = "stop"`). Good for logging the final
answer, persisting the transcript, or emitting metrics about turn length.


In [ ]:

hooks.fire(HookEvent::Stop, &HookPayload::Stop { reason: "stop".into() });


### 5.5 & 5.6 · `subagentstart` / `subagentstop`

Fired around spawning and finishing a sub-agent — the bracket around any
delegated child task. Use them to track fan-out, tag spans, or sum sub-costs.


In [ ]:

hooks.fire(HookEvent::SubAgentStart, &HookPayload::SubAgentStart {
    name: "code_reviewer".into(),
    prompt: "Review the latest diff for correctness.".into(),
});
hooks.fire(HookEvent::SubAgentStop, &HookPayload::SubAgentStop {
    name: "code_reviewer".into(),
    result: "2 issues found: unused import, unhandled panic.".into(),
});


### The ordered log — proof the hooks fired

Because we enabled the shared log, here is the exact chronological sequence of
every hook that fired, in the order it happened. This is what an observability
pipeline would collect:


In [ ]:

for (i, entry) in hooks.log_entries().iter().enumerate() {
    println!("{:2}. {}", i + 1, entry.event.as_str());
}


### Bonus: see the hooks fire inside a real agent run

If you have a reachable endpoint, the cell below runs the actual `Agent::run`
loop with the registered hooks wired in, then replays the hook log. If no
endpoint is reachable it prints a note instead. (Try it once you've set
`OPENAI_BASE_URL`.)


In [ ]:

// Wire the same hooks into a live agent and watch them fire in context.
use agent_loop::tools::ToolRegistry;
use agent_loop::chat::{ChatClient, default_model};
use agent_loop::{Agent, AgentConfig};

let registry = ToolRegistry::with_builtins();
let client = ChatClient::from_env();
let mut agent = Agent::new(AgentConfig::new(
    default_model(),
    "You are a terse assistant. Prefer bash for computations.",
    registry,
    hooks.clone(),
    client,
));
agent.config.parallel_tools = true;

match agent.run("Run `echo hooked` in the shell and tell me the output.") {
    Ok(outcome) => {
        println!("final answer: {}", outcome.final_text);
        println!("\nhook order: {}", agent.config.hooks.log_entries()
            .iter().map(|e| e.event.as_str()).collect::<Vec<_>>().join(" → "));
    }
    Err(e) => println!("[no reachable endpoint] live run skipped: {e}"),
}
